# Check Vermögensverzeichnis correction algorithm

Goal: measure how the `correct_vermogenverzeichnis_prediction` algorithm behaves on real production predictions.

Steps:
1. Fetch `is_va = True` predictions (sample size = 5000) together with their cleaned texts (from `GPUTask`).
2. Apply the correction algorithm to each text.
3. Collect into a separate dataframe all cases where the LLM said *va* but the correction algorithm flipped it to *not va*.

In [1]:
import os
import sys
from configparser import RawConfigParser

import pandas as pd

# make the aftercourt_automation package importable (utils.*)
PROJECT_DIR = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation"
if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)

# make the intent_recognition package importable (src.* modules)
INTENT_RECOG_DIR = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/intent_recognition"
if INTENT_RECOG_DIR not in sys.path:
    sys.path.append(INTENT_RECOG_DIR)

from src.db.database_manager import DatabaseManager, session_scope
from src.graph.repository.orm import GPUTask

# read the graph DB URI from secret.ini
_config = RawConfigParser()
_config.read(os.path.expanduser("~/secret.ini"))
GRAPH_DATABASE_URI = _config.get("GRAPH_DB", "GRAPH_DATABASE_URI")

graph_db_manager = DatabaseManager(db_uri=GRAPH_DATABASE_URI)

2026-06-17 17:18:55.976 | INFO     | src.db.database_manager:__init__:17 - Using provided DB URI.


In [13]:
# Correction algorithm (defined inline here so the notebook is self-contained and easy to iterate on).
import copy
import re

from loguru import logger

# ---- regexes ----------------------------------------------------------------
PROTOKOLL_DOC_REGEX = re.compile(
    r"(?im)^\s*(?:"
    r"protokoll"
    r"|verm[oö0]gens\s*auskunfts?\s*protokoll"
    r")\b"
)

VERMOEGENSVERZEICHNIS_TITLE_REGEX = re.compile(
    r"(?im)^\s*verm[oö0]gens\s*verzeichnis\b"
)

VERMOEGENSVERZEICHNIS_FORM_FIELDS_REGEX = re.compile(
    r"(?im)^\s*(?:"
    r"vorname(?:\(n\)|n)?"
    r"|rufname"
    r"|titel"
    r"|fahrzeuge"
    r"|geschlecht"
    r"|geburtsname"
    r"|bargeld"
    r"|wohnungseinrichtung"
    r"|haushaltsw[äa]sche"
    r"|wertpapiere"
    r"|geburtsdatum"
    r"|anschrift"
    r"|familienstand"
    r")\s*:?",
)

# minimum number of vermogenverzeichnis form fields required to consider a
# protokoll document a real vermogenverzeichnis
MIN_VERMOEGENSVERZEICHNIS_FORM_FIELDS = 6


# ---- correction function ----------------------------------------------------
def correct_vermogenverzeichnis_prediction(
    processed_model_output: dict | None,
    text: str | None,
    gpu_task_uuid: str | None = None,
) -> dict | None:
    """Precision-targeted correction for vermogenverzeichnis prediction based on document structure and keywords.

    Returns a corrected copy of ``processed_model_output`` (the input is not mutated).
    """
    if not processed_model_output or not text:
        return processed_model_output
    is_va = processed_model_output.get("is_va", False)
    if not is_va:
        return processed_model_output

    corrected = copy.deepcopy(processed_model_output)

    # Algorithm 1: protokoll without vermogenverzeichnis title or too few form fields -> not va
    if True:
        has_title = bool(VERMOEGENSVERZEICHNIS_TITLE_REGEX.search(text))
        has_enough_fields = (
            len(VERMOEGENSVERZEICHNIS_FORM_FIELDS_REGEX.findall(text)) >= MIN_VERMOEGENSVERZEICHNIS_FORM_FIELDS
        )
        if not (has_title and has_enough_fields):
            corrected["is_va"] = False
            logger.info(
                "Vermogenverzeichnis prediction corrected to False via Algorithm 1 "
                "(Protokoll without Vermogenverzeichnis title or fewer than {} form fields): GPUTask[{}]",
                MIN_VERMOEGENSVERZEICHNIS_FORM_FIELDS,
                gpu_task_uuid,
            )
            return corrected

    return corrected

In [3]:
# 1) Fetch is_va=True predictions (sample size = 5000) together with their texts.
SAMPLE_SIZE = 5000

# JSON filter: processed_model_output ->> 'is_va' == 'true'
is_va_filter = GPUTask.processed_model_output["is_va"].as_boolean() == True  # noqa: E712

with session_scope(graph_db_manager) as session:
    rows = (
        session.query(GPUTask)
        .filter(GPUTask.model_name == "vermogenverzeichnis_egvp")
        .filter(is_va_filter)
        .order_by(GPUTask.created_at.desc())
        .limit(SAMPLE_SIZE)
        .all()
    )
    va_df = pd.DataFrame(
        [{c.name: getattr(row, c.name) for c in GPUTask.__table__.columns} for row in rows]
    )

print("fetched rows:", len(va_df))
va_df.head()

fetched rows: 85


,gpu_task_uuid,task_uuid,ticket_uuid,model_name,status,model_input,model_output,processed_model_output,retry_count,attachment_id,meta_info,created_at,updated_at
0,e5219fb3-fb7e-4419-b40d-1df4fb89cca3,a3343d7a-e1f5-4f64-be2c-8aa01132701e,f505ff83-4c06-521e-bedf-29f862051481,vermogenverzeichnis_egvp,done,{'text': 'Anlage zur Niederschrift d.: GV'in M...,"{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,69000137,"{'instance': {'private_ip': '100.2.36.225', 'i...",2026-06-17 14:25:38.661544,2026-06-17 14:43:16.146608
1,6e144dd6-b833-4d5e-bb6a-60f943929c76,e73334ad-06fc-4a35-9aaa-6c648704c7a6,b4097886-f05b-5c09-b26a-60c49128db38,vermogenverzeichnis_egvp,done,{'text': 'Anlage zur Niederschrift d.: GV (b) ...,"{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,69000093,"{'instance': {'private_ip': '100.2.36.225', 'i...",2026-06-17 14:25:36.266140,2026-06-17 14:42:55.676965
2,646485ca-a8b3-418f-8cf6-6e456cfc1549,8d348998-b835-48d6-bab9-e0d5b43116a8,32c5f3fa-b1f2-5424-88e6-ff7ad8ba349e,vermogenverzeichnis_egvp,done,{'text': 'Anlage zur Niederschrift d. Gerichts...,"{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,68998515,"{'instance': {'private_ip': '100.2.36.225', 'i...",2026-06-17 13:24:31.208542,2026-06-17 13:40:53.285592
3,ed43d92c-4f71-41a5-bb94-59f96a5be5a4,ddf5416b-bf4e-4e49-ba78-8df3af63f5b6,085e4400-ab7d-5d4e-9bc7-009bca3f5b7b,vermogenverzeichnis_egvp,done,{'text': 'Anlage zur Niederschrift d. Obergeri...,"{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,68997459,"{'instance': {'private_ip': '100.2.36.1', 'ins...",2026-06-17 12:25:44.675233,2026-06-17 12:26:59.147567
4,7656db06-2b91-4531-aa89-9a9001ce4d4f,8e530b94-c84c-48d2-8f23-9625f7039661,7bc72b90-620c-52cc-b0eb-c5bcf2ba78c4,vermogenverzeichnis_egvp,done,{'text': 'Anlage zur Niederschrift d. Obergeri...,"{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,68991360,"{'instance': {'private_ip': '100.2.36.1', 'ins...",2026-06-17 09:24:38.930443,2026-06-17 09:48:42.675475


In [4]:
va_df.shape

(85, 13)

In [5]:
# Extract the cleaned text used by the model (model_input['clean_text']).
def _extract_clean_text(model_input):
    if isinstance(model_input, dict):
        return model_input.get("clean_text") or ""
    return ""

va_df["text"] = va_df["model_input"].map(_extract_clean_text)

# sanity check: how many have non-empty text
print("rows with non-empty text:", (va_df["text"].str.len() > 0).sum())
va_df[["gpu_task_uuid", "attachment_id", "processed_model_output"]].head()

rows with non-empty text: 85


,gpu_task_uuid,attachment_id,processed_model_output
0,e5219fb3-fb7e-4419-b40d-1df4fb89cca3,69000137,{'is_va': True}
1,6e144dd6-b833-4d5e-bb6a-60f943929c76,69000093,{'is_va': True}
2,646485ca-a8b3-418f-8cf6-6e456cfc1549,68998515,{'is_va': True}
3,ed43d92c-4f71-41a5-bb94-59f96a5be5a4,68997459,{'is_va': True}
4,7656db06-2b91-4531-aa89-9a9001ce4d4f,68991360,{'is_va': True}


In [14]:
# 2) Apply the correction algorithm to each text.
def _apply_correction(row):
    corrected = correct_vermogenverzeichnis_prediction(
        processed_model_output=row["processed_model_output"],
        text=row["text"],
        gpu_task_uuid=row["gpu_task_uuid"],
    )
    return bool(corrected.get("is_va", False)) if corrected else False

va_df["corrected_is_va"] = va_df.apply(_apply_correction, axis=1)

# True  -> still vermogenverzeichnis after correction
# False -> algorithm flipped it to NOT vermogenverzeichnis
va_df["corrected_is_va"].value_counts()

2026-06-17 17:27:23.842 | INFO     | __main__:correct_vermogenverzeichnis_prediction:68 - Vermogenverzeichnis prediction corrected to False via Algorithm 1 (Protokoll without Vermogenverzeichnis title or fewer than 6 form fields): GPUTask[21e21273-b414-4d56-bd1a-66d6e449d059]
2026-06-17 17:27:23.845 | INFO     | __main__:correct_vermogenverzeichnis_prediction:68 - Vermogenverzeichnis prediction corrected to False via Algorithm 1 (Protokoll without Vermogenverzeichnis title or fewer than 6 form fields): GPUTask[66d91919-956c-40d8-918b-840634e02314]
2026-06-17 17:27:23.846 | INFO     | __main__:correct_vermogenverzeichnis_prediction:68 - Vermogenverzeichnis prediction corrected to False via Algorithm 1 (Protokoll without Vermogenverzeichnis title or fewer than 6 form fields): GPUTask[c912c82c-0b62-4b57-9061-b8e45f190234]
2026-06-17 17:27:23.847 | INFO     | __main__:correct_vermogenverzeichnis_prediction:68 - Vermogenverzeichnis prediction corrected to False via Algorithm 1 (Protokoll wi

corrected_is_va
True     66
False    19
Name: count, dtype: int64

In [15]:
# 3) Separate dataframe of cases the algorithm flipped: LLM said va, correction says NOT va.
corrected_df = va_df[~va_df["corrected_is_va"]].reset_index(drop=True)

n_total = len(va_df)
n_flipped = len(corrected_df)
print(f"LLM is_va=True samples : {n_total}")
print(f"flipped to NOT va      : {n_flipped} ({(n_flipped / n_total * 100) if n_total else 0:.1f}%)")

corrected_df[["gpu_task_uuid", "attachment_id", "created_at"]]

LLM is_va=True samples : 85
flipped to NOT va      : 19 (22.4%)


,gpu_task_uuid,attachment_id,created_at
0,21e21273-b414-4d56-bd1a-66d6e449d059,68869968,2026-06-16 15:25:16.505751
1,66d91919-956c-40d8-918b-840634e02314,68860852,2026-06-16 09:23:54.553557
2,c912c82c-0b62-4b57-9061-b8e45f190234,68714756,2026-06-15 16:13:46.724462
3,a20178c9-b35e-43c4-88a2-73078cbb1e8e,68710652,2026-06-15 13:27:43.184615
4,b361b66c-44bb-4152-b484-3a0f154cb14b,68700106,2026-06-15 09:27:57.125441
5,1ac93852-8b41-46d1-a7eb-868f2a49c15a,68361892,2026-06-13 12:22:24.760007
6,a8cc15d5-91d4-44c1-9301-e1ce3acab262,68336179,2026-06-12 15:19:05.309572
7,2620bf27-814c-45e7-b628-240d735c4495,68312784,2026-06-11 17:22:31.326877
8,816c67bb-2445-47d8-afef-5329cd4fd1e9,68309691,2026-06-11 13:26:24.492300
9,0538265d-8d4c-46ac-bef3-4067706028be,68306250,2026-06-11 10:28:04.173635


In [16]:
# Diagnostic: label which algorithm caused the flip (to see *how* the correction works).
def _flip_reason(text):
    if not text:
        return "no_text"
    # Algorithm 1: protokoll without title or too few form fields
    if PROTOKOLL_DOC_REGEX.search(text):
        has_title = bool(VERMOEGENSVERZEICHNIS_TITLE_REGEX.search(text))
        n_fields = len(VERMOEGENSVERZEICHNIS_FORM_FIELDS_REGEX.findall(text))
        if not has_title:
            return "algo1_protokoll_no_title"
        if n_fields < MIN_VERMOEGENSVERZEICHNIS_FORM_FIELDS:
            return f"algo1_protokoll_few_fields({n_fields})"
    return "unknown"

corrected_df["flip_reason"] = corrected_df["text"].map(_flip_reason)
corrected_df["flip_reason"].value_counts()

flip_reason
algo1_protokoll_no_title         15
unknown                           3
algo1_protokoll_few_fields(3)     1
Name: count, dtype: int64

In [17]:
# Inspect a single flipped case (change the index to review different documents).
idx = 0
if len(corrected_df):
    row = corrected_df.iloc[idx]
    print("gpu_task_uuid:", row["gpu_task_uuid"])
    print("attachment_id:", row["attachment_id"])
    print("flip_reason  :", row["flip_reason"])
    print("=" * 80)
    print(row["text"])
else:
    print("No flipped cases to inspect.")

gpu_task_uuid: 21e21273-b414-4d56-bd1a-66d6e449d059
attachment_id: 68869968
flip_reason  : algo1_protokoll_no_title
Marc-Andre Steurer
88167 Gestratz, den
Obergerichtsvollzieher beim
SPRECHSTUNDEN:
Protokoll
Amtsgericht Lindau (Bodensee)
Mo. und Mi. 16 17 Uhr
Gewerbepark Edelweiss Nr. 4
Auftrag des Gläub.
durchgeführte
88138 Weißensberg
Amtshandlungen
Tel: 08389/2970161
Fax: AG Li 08382/2607501
Tel:
Vermögensauskunft
Vermögensauskunft
e-Mail: gvsteurer@kabelbw.de
Pfändung
Pfändung
IBAN: DE92 6609 0800 0003 0845 40 - BIC: GENODE61BBB
Drittstellenauskünfte
Drittstellenauskünfte
5 DR-II 0455/26
Aufenthaltsermittlung
Aufenthaltsermittlung
Bitte bei allen Schreiben angeben
Verhaftung
Verhaftung
Gütliche Erledigung
Gütliche Erledigung
OGV Marc-Andre Steurer, Gewerbepark Edelweiss Nr. 4. 88138 Weißensberg
Pair Finance GmbH
Outfittery GmbH, 10961 Berlin
10.06.26
gegen
Gläub.-Vertr.
Knesebeckstraße 62-63
Fimpel Andre, Altenburg 38A, 88167 Gestratz
10719 Berlin
Schuldtitel
Vollstreckungsbescheid

In [18]:
corrected_df

,gpu_task_uuid,task_uuid,ticket_uuid,model_name,status,model_input,model_output,processed_model_output,retry_count,attachment_id,meta_info,created_at,updated_at,text,corrected_is_va,flip_reason
0,21e21273-b414-4d56-bd1a-66d6e449d059,a5ad68ad-3be5-4797-94dc-245f78a5481f,3f0f3390-e8a9-5437-8b52-f7fcb1e95306,vermogenverzeichnis_egvp,done,"{'text': 'Marc-Andre Steurer 88167 Gestratz, d...","{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,68869968,"{'instance': {'private_ip': '100.2.38.122', 'i...",2026-06-16 15:25:16.505751,2026-06-16 15:45:45.118360,"Marc-Andre Steurer\n88167 Gestratz, den\nOberg...",False,algo1_protokoll_no_title
1,66d91919-956c-40d8-918b-840634e02314,118dc860-7486-48c7-8751-93b17fb0c3b0,5006a88d-10e8-5c8b-ba5a-4b3bad5beaca,vermogenverzeichnis_egvp,done,{'text': 'Obergerichtsvollzieherin Am Markt 8 ...,"{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,68860852,"{'instance': {'private_ip': '100.2.38.122', 'i...",2026-06-16 09:23:54.553557,2026-06-16 09:28:33.329322,Obergerichtsvollzieherin\nAm Markt 8\nKatja Ba...,False,algo1_protokoll_no_title
2,c912c82c-0b62-4b57-9061-b8e45f190234,2680211c-565e-4269-ba7f-31b07910c0f9,6683f6fb-3102-5eb8-9797-d41302ce0b7f,vermogenverzeichnis_egvp,done,{'text': 'Doreen Hayashi Obergerichtsvollziehe...,"{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,68714756,"{'instance': {'private_ip': '100.2.36.225', 'i...",2026-06-15 16:13:46.724462,2026-06-15 16:49:35.927496,Doreen Hayashi\nObergerichtsvollzieherin\nAn d...,False,algo1_protokoll_no_title
3,a20178c9-b35e-43c4-88a2-73078cbb1e8e,3351642c-7ce4-47db-984a-3ebaa48ed900,b5fa94c8-8843-5eba-afd5-ce36775e8fc5,vermogenverzeichnis_egvp,done,{'text': 'Bitte stets angeben: DR II 334/26 is...,"{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,68710652,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-15 13:27:43.184615,2026-06-15 13:59:24.929830,Bitte stets angeben:\nDR II 334/26\nist elektr...,False,algo1_protokoll_no_title
4,b361b66c-44bb-4152-b484-3a0f154cb14b,84f66d56-f6fb-492e-af3e-d26714be5355,e89d5bd3-27d9-5ac0-bee8-70d07d98f534,vermogenverzeichnis_egvp,done,{'text': 'Die nachstehend gewählten Formulieru...,"{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,68700106,"{'instance': {'private_ip': '100.2.36.225', 'i...",2026-06-15 09:27:57.125441,2026-06-15 09:53:20.247925,Die nachstehend gewählten Formulierungen gelte...,False,algo1_protokoll_no_title
5,1ac93852-8b41-46d1-a7eb-868f2a49c15a,f8d5ca90-5977-445d-83cf-2e9ec541334c,8dbfaeb1-8c27-57ae-b6e8-48fc668a80f3,vermogenverzeichnis_egvp,done,{'text': 'SABINE LOTZE Die nachstehend gewählt...,"{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,68361892,"{'instance': {'private_ip': '100.2.38.65', 'in...",2026-06-13 12:22:24.760007,2026-06-13 12:57:43.647317,SABINE LOTZE\nDie nachstehend gewählten Formul...,False,algo1_protokoll_no_title
6,a8cc15d5-91d4-44c1-9301-e1ce3acab262,0ac8ade2-10fb-4f0f-9dc0-a107dcd9c9bb,66df814a-2169-57fb-986f-55c06bb6f9ad,vermogenverzeichnis_egvp,done,{'text': 'Dominic Schulz Heiligkreuzgasse 34 G...,"{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,68336179,"{'instance': {'private_ip': '100.2.36.225', 'i...",2026-06-12 15:19:05.309572,2026-06-12 15:39:12.878964,Dominic Schulz\nHeiligkreuzgasse 34\nGerichtsv...,False,algo1_protokoll_few_fields(3)
7,2620bf27-814c-45e7-b628-240d735c4495,210d7269-edf1-423e-8704-630ca5e260b3,904f04d5-2a3a-5128-8f8e-28820ab16b70,vermogenverzeichnis_egvp,done,{'text': 'Martina Breloer Die nachstehend gewä...,"{'answer': '{""is_va"": true}', 'answer_reasonin...",{'is_va': True},0,68312784,"{'instance': {'private_ip': '100.2.38.122', 'i...",2026-06-11 17:22:31.326877,2026-06-11 17:27:57.079568,Martina Breloer\nDie nachstehend gewählten For...,False,algo1_protokoll_no_title
8,816c67bb-2445-47d8-afef-5329cd4fd1e9,e7693ad8-b4e6-4aed-9773-77a3086c1b05,5a85ae23-ee50-599b-97e3-af40f4516

In [11]:
# Download PDFs for all corrected (flipped) cases.
import boto3
from IPython.display import clear_output
from python_utilities.db_connection import DbConnection

from utils.prod_utils import get_data_by_attachment_id

DOWNLOAD_DIR = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/tmp_va_corrected_check"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

analytics_db = DbConnection("ANALYTICS", "PROD_RDS")
_boto_session = boto3.Session(profile_name="739275445236_DataScienceUser")
s3 = _boto_session.client("s3")

attachment_ids = corrected_df["attachment_id"].dropna().unique().tolist()
print(f"Downloading PDFs for {len(attachment_ids)} corrected attachment(s) → {DOWNLOAD_DIR}\n")

for i, a_id in enumerate(attachment_ids, 1):
    get_data_by_attachment_id(
        a_id,
        analytics_db,
        s3,
        pdf_download=True,
        pdf_download_dir=DOWNLOAD_DIR,
        verbose=False,
    )
    clear_output(wait=True)
    print(f"[{i}/{len(attachment_ids)}] downloaded attachment_id={a_id}")

print("\nDone.")

[16/16] downloaded attachment_id=67493020

Done.


In [19]:
# Download PDFs where flip_reason == "unknown" to a dedicated sub-folder.
UNKNOWN_DOWNLOAD_DIR = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/tmp_va_corrected_check/unknown"
os.makedirs(UNKNOWN_DOWNLOAD_DIR, exist_ok=True)

unknown_ids = (
    corrected_df[corrected_df["flip_reason"] == "unknown"]["attachment_id"]
    .dropna()
    .unique()
    .tolist()
)
print(f"Downloading PDFs for {len(unknown_ids)} 'unknown' attachment(s) → {UNKNOWN_DOWNLOAD_DIR}\n")

for i, a_id in enumerate(unknown_ids, 1):
    get_data_by_attachment_id(
        a_id,
        analytics_db,
        s3,
        pdf_download=True,
        pdf_download_dir=UNKNOWN_DOWNLOAD_DIR,
        verbose=False,
    )
    clear_output(wait=True)
    print(f"[{i}/{len(unknown_ids)}] downloaded attachment_id={a_id}")

print("\nDone.")


[3/3] downloaded attachment_id=68135139

Done.
